In [1]:
import xml.etree.ElementTree as ET
import requests
import csv
import os
import fitz  # PyMuPDF
from pdf2image import convert_from_bytes
import pytesseract
from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import re
import json
import base64
import hashlib
import threading

# Constants
OAI_BASE = "https://digital.library.unt.edu/oai/"
COLLECTION_SET = "collection:IIPCM"
NAMESPACES = {
    "oai": "http://www.openarchives.org/OAI/2.0/",
    "untl": "http://digital2.library.unt.edu/untl/",
}
OUTPUT_FILE = "iipcm_extracted_content.csv"
MAX_WORKERS = 10  # Speed up harvesting via thread pool

# Create a single requests Session to maintain session cookies
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
})

altcha_lock = threading.Lock()

def safe_get(url):
    """Wrapper for requests.get that automatically solves Altcha proof-of-work challenges if triggered."""
    r = session.get(url, timeout=15)
    
    if "Gauging your humanity" in r.text:
        with altcha_lock:
            # Re-verify inside the lock in case another thread solved it in the meantime
            r = session.get(url, timeout=15)
            if "Gauging your humanity" not in r.text:
                return r
                
            print(f"\n🔒 Altcha challenge triggered for: {url}. Solving...")
            try:
                # Extract CSRF token
                csrf_match = re.search(r'name="csrfmiddlewaretoken"\s+value="([^"]+)"', r.text)
                if not csrf_match:
                    print("❌ Could not find CSRF token on Altcha page.")
                    return r
                csrf_token = csrf_match.group(1)
                
                # Extract challenge variables using regex
                algo_match = re.search(r'algorithm:\s*"([^"]+)"', r.text)
                challenge_match = re.search(r'challenge:\s*"([^"]+)"', r.text)
                salt_match = re.search(r'salt:\s*"([^"]+)"', r.text)
                sig_match = re.search(r'signature:\s*"([^"]+)"', r.text)
                max_match = re.search(r'maxnumber:\s*"([^"]+)"', r.text)
                
                if not all([algo_match, challenge_match, salt_match, sig_match, max_match]):
                    print("❌ Could not extract challenge details.")
                    return r
                    
                algorithm = algo_match.group(1)
                challenge_hash = challenge_match.group(1)
                salt = salt_match.group(1)
                signature = sig_match.group(1)
                maxnumber = int(max_match.group(1))
                
                # Solve SHA-256 puzzle
                solved_num = None
                for n in range(maxnumber + 1):
                    h = hashlib.sha256((salt + str(n)).encode()).hexdigest()
                    if h == challenge_hash:
                        solved_num = n
                        break
                        
                if solved_num is None:
                    print("❌ Failed to solve the Altcha puzzle.")
                    return r
                    
                # Build base64 payload expected by Altcha
                altcha_payload = {
                    "algorithm": algorithm,
                    "challenge": challenge_hash,
                    "number": solved_num,
                    "salt": salt,
                    "signature": signature
                }
                altcha_payload_str = json.dumps(altcha_payload, separators=(',', ':'))
                altcha_b64 = base64.b64encode(altcha_payload_str.encode()).decode()
                
                # Submit verified token to establish session
                submit_url = "https://digital.library.unt.edu/dam/submit/"
                headers = {
                    "X-CSRFToken": csrf_token,
                    "Origin": "https://digital.library.unt.edu",
                    "Referer": url
                }
                payload_data = {
                    "csrfmiddlewaretoken": csrf_token,
                    "altcha": altcha_b64
                }
                
                submit_resp = session.post(submit_url, data=payload_data, headers=headers, timeout=15)
                if submit_resp.status_code == 200 and "success" in submit_resp.text:
                    print("🔓 Altcha solved and session authenticated successfully!")
                    # Retry the original download
                    r = session.get(url, timeout=15)
                else:
                    print(f"❌ Failed to submit Altcha: {submit_resp.text}")
            except Exception as e:
                print(f"❌ Error during Altcha resolution: {e}")
                
    return r

def extract_pdf_text(pdf_url):
    try:
        response = safe_get(pdf_url)
        if response.status_code != 200 or "application/pdf" not in response.headers.get("Content-Type", ""):
            return ""
        pdf_bytes = response.content

        # Try extracting using PyMuPDF
        try:
            doc = fitz.open(stream=pdf_bytes, filetype="pdf")
            text = "\n".join([page.get_text() for page in doc])
            if text.strip():
                return text.strip()
        except Exception:
            pass

        # Fallback to OCR
        images = convert_from_bytes(pdf_bytes)
        text = ""
        for img in images:
            text += pytesseract.image_to_string(img) + "\n"
        return text.strip()

    except Exception:
        return ""

def extract_vtt_transcript(vtt_url):
    try:
        response = safe_get(vtt_url)
        response.raise_for_status()
        lines = response.text.splitlines()
        transcript = []

        for line in lines:
            line = line.strip()
            if (
                line.startswith("WEBVTT") or
                line.startswith("NOTE") or
                "-->" in line or
                line.isdigit() or
                (":" in line and line.lower().startswith("vtt_")) or
                line == ""
            ):
                continue
            transcript.append(line)

        return " ".join(transcript)

    except Exception as e:
        print(f"❌ Error parsing VTT from {vtt_url}: {e}")
        return ""

def harvest_oai_records():
    print("🔁 Harvesting metadata using: untl")
    records = []
    token = None

    while True:
        params = {
            "verb": "ListRecords",
            "metadataPrefix": "untl",
            "set": COLLECTION_SET
        } if not token else {
            "verb": "ListRecords",
            "resumptionToken": token
        }

        resp = safe_get(OAI_BASE + "?" + "&".join([f"{k}={v}" for k, v in params.items()]))
        root = ET.fromstring(resp.content)

        for r in root.findall(".//oai:record", NAMESPACES):
            header = r.find("oai:header", NAMESPACES)
            if header is None or header.attrib.get("status") == "deleted":
                continue

            meta = r.find(".//untl:metadata", NAMESPACES)
            if meta is None:
                continue

            # Item URL (ark_url)
            item_url_el = meta.find('.//untl:identifier[@qualifier="itemURL"]', NAMESPACES)
            ark_url = item_url_el.text.strip() if item_url_el is not None and item_url_el.text else ""
            if not ark_url:
                continue

            # Title
            title_el = meta.find("untl:title", NAMESPACES)
            title = title_el.text.strip() if title_el is not None and title_el.text else ""

            # Date
            date_el = meta.find("untl:date", NAMESPACES)
            date = date_el.text.strip() if date_el is not None and date_el.text else ""

            # Creators (with affiliations)
            creators = []
            for creator_el in meta.findall("untl:creator", NAMESPACES):
                name_el = creator_el.find("untl:name", NAMESPACES)
                info_el = creator_el.find("untl:info", NAMESPACES)
                name = name_el.text.strip() if name_el is not None and name_el.text else ""
                info = info_el.text.strip() if info_el is not None and info_el.text else ""
                if name:
                    if info:
                        creators.append(f"{name} ({info})")
                    else:
                        creators.append(name)
            creator_str = "; ".join(creators)

            # Subjects
            subjects = [el.text.strip() for el in meta.findall("untl:subject", NAMESPACES) if el.text]
            subject_str = "; ".join(subjects)

            # Description
            desc_el = meta.find("untl:description", NAMESPACES)
            description = desc_el.text.strip() if desc_el is not None and desc_el.text else ""

            # Conference details
            conf_el = meta.find('.//untl:source[@qualifier="conference"]', NAMESPACES)
            conference = conf_el.text.strip() if conf_el is not None and conf_el.text else ""

            if conference:
                description = f"{description}\nConference: {conference}"

            # Resource Type
            type_el = meta.find("untl:resourceType", NAMESPACES)
            item_type = type_el.text.strip() if type_el is not None and type_el.text else "text"

            record = {
                "ark_url": ark_url,
                "title": title,
                "date": date,
                "creator": creator_str,
                "subject": subject_str,
                "description": description,
                "item_type": item_type,
            }

            records.append(record)

        token_el = root.find(".//oai:resumptionToken", NAMESPACES)
        token = token_el.text.strip() if token_el is not None and token_el.text else None
        if not token:
            break

    print(f"✅ Retrieved {len(records)} metadata records.\n")
    return records

def resolve_pdf_link(folder_url):
    try:
        res = safe_get(folder_url)
        soup = BeautifulSoup(res.text, "html.parser")
        for link in soup.find_all("a"):
            href = link.get("href", "")
            if href.endswith(".pdf"):
                return requests.compat.urljoin(folder_url, href)
    except Exception as e:
        print(f"❌ Error resolving PDF from folder: {folder_url} - {e}")
    return ""

def process_record(record):
    ark_url = record["ark_url"]
    item_type = record["item_type"].lower()
    text = ""
    file_url = ""

    try:
        res = safe_get(ark_url)
        soup = BeautifulSoup(res.text, "html.parser")
        links = soup.find_all("a")

        for link in links:
            href = link.get("href", "")
            full_url = requests.compat.urljoin(ark_url, href)

            if "video" in item_type and href.endswith(".vtt"):
                text = extract_vtt_transcript(full_url)
                file_url = full_url
                break

            # Match direct PDF links, or folders containing high-res PDFs
            elif "video" not in item_type and (href.endswith(".pdf") or "high_res_d/" in href or "/m2/" in href):
                if href.endswith("/"):
                    pdf_resolved = resolve_pdf_link(full_url)
                    if pdf_resolved:
                        text = extract_pdf_text(pdf_resolved)
                        file_url = pdf_resolved
                        break
                else:
                    text = extract_pdf_text(full_url)
                    file_url = full_url
                    break

    except Exception:
        pass

    record["source_url"] = file_url
    record["full_text"] = text.replace("\r", "").replace("\n", "\\n")
    return record

def main():
    # 1. Harvest metadata using untl prefix
    records = harvest_oai_records()
    
    # 2. Check for existing file to support incremental / resumable harvesting
    existing_records = {}
    if os.path.exists(OUTPUT_FILE):
        print(f"📂 Found existing progress in {OUTPUT_FILE}. Loading already processed documents...")
        try:
            with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    if row.get("ark_url") and row.get("full_text"):
                        existing_records[row["ark_url"]] = row
            print(f"✅ Loaded {len(existing_records)} already completed documents. These will be skipped.")
        except Exception as e:
            print(f"⚠️ Failed to load existing file, starting fresh: {e}")

    print("🔍 Extracting full text from associated files (using ThreadPool)... \n")
    processed = []
    to_process = []
    
    for rec in records:
        url = rec["ark_url"]
        if url in existing_records:
            processed.append(existing_records[url])
        else:
            to_process.append(rec)
            
    print(f"📋 Total: {len(records)} | Skipped: {len(processed)} | To Process: {len(to_process)}")

    if to_process:
        # Use ThreadPoolExecutor to harvest/OCR documents concurrently
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(process_record, rec): rec for rec in to_process}
            
            # Progress bar for parallel threads
            for future in tqdm(as_completed(futures), total=len(to_process), desc="📄 Processing"):
                try:
                    res = future.result()
                    processed.append(res)
                except Exception as e:
                    rec = futures[future]
                    print(f"❌ Error processing {rec['ark_url']}: {e}")
                    rec["source_url"] = ""
                    rec["full_text"] = ""
                    processed.append(rec)

    keys = ["ark_url", "title", "date", "creator", "subject", "description", "item_type", "source_url", "full_text"]
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(processed)

    print(f"\n✅ Exported {len(processed)} records with extracted content to: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()


🔁 Harvesting metadata using: untl
✅ Retrieved 626 metadata records.

🔍 Extracting full text from associated files (using ThreadPool)... 

📋 Total: 626 | Skipped: 0 | To Process: 626


📄 Processing:   0%|          | 0/626 [00:00<?, ?it/s]


🔒 Altcha challenge triggered for: https://digital.library.unt.edu/ark:/67531/metadc1476385/. Solving...
🔓 Altcha solved and session authenticated successfully!


📄 Processing: 100%|██████████| 626/626 [22:55<00:00,  2.20s/it]


✅ Exported 626 records with extracted content to: iipcm_extracted_content.csv
